In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import layers

from PIL import Image,ImageDraw,ImageFont

from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

In [3]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [4]:

def getSpecificExtensionFiles(path, extension):
    out = []
    for (path, dir, files) in os.walk(path):
        for filename in files:
            ext = os.path.splitext(filename)[-1]
            if ext == extension:
                #print("%s/%s" % (path, filename))
                out.append(path + "/" + filename)
    return out
#print(files)

# for file in files:
#     newName = file
#     newName = newName.replace(" ", "")
#     print(newName)
#     os.rename(file, newName)

        

In [5]:
import matplotlib.pyplot as plt
import random

# draw_text = '람'
# font = "/root/Data/font/clova-all/가람연꽃/나눔손글씨_가람연꽃.ttf"

# fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")

from PIL import Image,ImageDraw,ImageFont

def CreateFontImage(str, fontPath):
    font = ImageFont.truetype(fontPath, 28, encoding = 'utf-8')
    left, top, right, bottom = font.getbbox(str)
    width = right - left
    height = bottom - top
    
    canvas = Image.new('RGB', (width + 10, height + 14), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((3,3), str, 'black', font)
    
    #print(canvas)
    img = tf.image.convert_image_dtype(canvas, tf.float32)
    img = tf.image.resize(img, (64, 64))    
    img = np.array(img)
    img = np.expand_dims(img, axis=0)
    
    #plt.imshow(img)
    #print(img)

    # save the blank canvas to a file
    #canvas.save("unicode-text.png", "PNG")
    #canvas.show()
    return img

    
#CreateFontImage(draw_text, font)

def getFontImage(fontPath, imageNum):
    #fontFiles = getSpecificExtensionFiles(fontPath, ".ttf")
    
    #for i in range(1, imageNum):
        #fontidx = random.randrange(0, len(fontFiles) + 1)
    cho = random.randrange(0, 19)
    jung = random.randrange(0, 21)
    jong = random.randrange(0, 28)
    
    ja = label2ja[cho]
    mo = label2mo[jung]
    ba = label2ba[jong]

    char = unicode.join_jamos_char(ja, mo ,ba)
    #print(char)
    
    label1 = np.expand_dims(np.array(cho), axis=0)
    label2 = np.expand_dims(np.array(jung), axis=0)
    label3 = np.expand_dims(np.array(jong), axis=0)
    

    #yield CreateFontImage(char, fontFiles[fontidx])
    return CreateFontImage(char, fontPath), (label1, label2, label3)

In [6]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-03-27 14:17:50.944864: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:50.966927: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:50.966973: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:51.148066: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:51.148163: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 316947723409031449
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 12658224430892454500
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [7]:
DATA_SIZE = 30000
VALID_DATA_SIZE = DATA_SIZE / 10

In [8]:
def get_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)
    
    os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("/root/Data/hangul/dataset/shuffled_tranDataset.csv", 'r', encoding='utf-8')
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        yield img, (label1, label2, label3)
        fontidx = random.randrange(0, len(fontFiles))
        
        #yield getFontImage(fontFiles[fontidx], 10)
        
        if (cnt > DATA_SIZE): break
        else                : cnt += 1
    
    
        
def get_valid_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)
    
    os.system("shuf /root/Data/hangul/dataset/validation.csv > /root/Data/hangul/dataset/shuffled_validation.csv")    
    csvFile = open("/root/Data/hangul/dataset/shuffled_validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)

        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        yield img, (label1, label2, label3)
        fontidx = random.randrange(0, len(fontFiles))
        
        #yield getFontImage(fontFiles[fontidx], 10)
        
        if (cnt > VALID_DATA_SIZE): break
        else                : cnt += 1

In [9]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

# print(dataset)
# iterator = iter(dataset)
# print(next(iterator))

2024-03-27 14:17:53.785337: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:53.785433: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:53.785465: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:53.785832: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:17:53.785865: I external/local_xla/xla/stream_executor

In [11]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

l = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(posts_input)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)


x = layers.Flatten()(l)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [12]:
#dataset = tf.data.Dataset.from_tensor_slices(({'input_x': data_a, 'input_y': data_b}, labels)).batch(2).repeat()
#model.fit(get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv"), epochs = 100, batch_size = 16)
#dataset = dataset.shuffle(150).batch(8)
model.fit(dataset, validation_data= validDtaset, epochs = 20, batch_size = 16)
#model.fit_generator(dataset, epochs = 100)


Epoch 1/20


I0000 00:00:1711516685.635999  123570 service.cc:145] XLA service 0x7f6038015110 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1711516685.636041  123570 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-03-27 14:18:05.696031: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-27 14:18:05.980173: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


     25/Unknown 6s 7ms/step - DenseCho2_accuracy: 0.0900 - DenseJong2_accuracy: 0.1526 - DenseJung2_accuracy: 0.0928 - loss: 67.9580  

I0000 00:00:1711516688.117700  123570 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  29996/Unknown 192s 6ms/step - DenseCho2_accuracy: 0.0898 - DenseJong2_accuracy: 0.1441 - DenseJung2_accuracy: 0.1043 - loss: 9.0493

...

30002/30002 ━━━━━━━━━━━━━━━━━━━━ 195s 7ms/step - DenseCho2_accuracy: 0.0899 - DenseJong2_accuracy: 0.1470 - DenseJung2_accuracy: 0.1073 - loss: 8.3532 - val_DenseCho2_accuracy: 0.0856 - val_DenseJong2_accuracy: 0.1542 - val_DenseJung2_accuracy: 0.1013 - val_loss: 8.3298
Epoch 4/20
   13/30002 ━━━━━━━━━━━━━━━━━━━━ 4:22 9ms/step - DenseCho2_accuracy: 0.2969 - DenseJong2_accuracy: 0.0917 - DenseJung2_accuracy: 0.0000e+00 - loss: 8.2458        

2024-03-27 14:27:53.618157: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-27 14:27:53.618201: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2024-03-27 14:27:53.618212: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10290264859373869693
2024-03-27 14:27:53.618217: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 5557414226931430739
2024-03-27 14:27:53.618222: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8340040256220788902
2024-03-27 14:27:53.618244: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3046429583699363216


14973/30002 ━━━━━━━━━━━━━━━━━━━━ 1:42 7ms/step - DenseCho2_accuracy: 0.0973 - DenseJong2_accuracy: 0.1566 - DenseJung2_accuracy: 0.0973 - loss: 8.3439

2024-03-27 14:29:35.557807: W tensorflow/core/framework/op_kernel.cc:1839] OP_REQUIRES failed at whole_file_read_ops.cc:116 : NOT_FOUND: /root/Data/hangul/image/040/04030011102.jpg03630009080.jpg; No such file or directory
2024-03-27 14:29:35.557863: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: NOT_FOUND: /root/Data/hangul/image/040/04030011102.jpg03630009080.jpg; No such file or directory
2024-03-27 14:29:35.559253: W tensorflow/core/framework/op_kernel.cc:1827] UNKNOWN: NotFoundError: {{function_node __wrapped__ReadFile_device_/job:localhost/replica:0/task:0/device:CPU:0}} /root/Data/hangul/image/040/04030011102.jpg03630009080.jpg; No such file or directory [Op:ReadFile]
Traceback (most recent call last):

  File "/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/tensorflow/python/ops/script_ops.py", line 270, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/anaconda3/envs/Anaconda_tensor/lib/pyth

UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  NotFoundError: {{function_node __wrapped__ReadFile_device_/job:localhost/replica:0/task:0/device:CPU:0}} /root/Data/hangul/image/040/04030011102.jpg03630009080.jpg; No such file or directory [Op:ReadFile]
Traceback (most recent call last):

  File "/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/tensorflow/python/ops/script_ops.py", line 270, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/tensorflow/python/data/ops/from_generator_op.py", l

In [11]:
model.save_weights('han_master5.weights.h5')

In [58]:
model.load_weights("han_master5.weights.h5")

img = tf.io.read_file("/root/Data/hangul_handwrite_val/image/test/35.jpg")
#img = tf.io.read_file("/root/Data/hangul_handwrite_val/image//31.jpg")
#img = tf.io.read_file("E:\\unzipData\\Training\\image_Training_handwrite\\1.letter\\040\\04030003097")
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.convert_image_dtype(img, tf.float32)
img = tf.image.resize(img, (64, 64))

print(img.shape)

img = np.array(img)

img = np.expand_dims(img, axis=0)

#print(img.shape)

ch, ju, jo = model.predict(img)

ch = ch.argmax()
ju = ju.argmax()
jo = jo.argmax()

ja = label2ja[ch]
mo = label2mo[ju]
ba = label2ba[jo]

char = unicode.join_jamos_char(ja, mo ,ba)
print(char)

(64, 64, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
솨
